# Dynamic Prompts (`@dynamic_prompt`)

A **dynamic prompt** lets you build the system prompt *at runtime* instead of hard-coding one string. The `@dynamic_prompt` middleware runs right before each model call and returns the system prompt to use for that call.

```
request (state + runtime context) --> @dynamic_prompt --> system prompt --> model
```

Typical uses: personalise by user, switch language, change persona by tier, or inject the current date / live data.

In [2]:
%run ../../langchain_common.py

C:\Users\akhawaja\git\cs4603\langchain_common.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
USER_AGENT environment variable not set, consider setting it to identify your requests.


## Personalise the prompt by customer tier

We pass a typed **runtime context** (`SupportContext`) when invoking the agent. The dynamic prompt reads that context and tailors the system prompt for each request.

In [3]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt, ModelRequest


@dataclass
class SupportContext:
    user_name: str
    tier: str  # "free" or "premium"


@dynamic_prompt
def personalized_prompt(request: ModelRequest) -> str:
    ctx = request.runtime.context
    base = "You are a customer-support assistant for an online store."
    if ctx.tier == "premium":
        return (
            f"{base} You are speaking with {ctx.user_name}, a PREMIUM customer. "
            "Be warm, thorough, and proactively offer extra help."
        )
    return (
        f"{base} You are speaking with {ctx.user_name}, a free-tier customer. "
        "Be polite but concise."
    )


agent = create_agent(
    model=llm_noreason,
    tools=[],
    context_schema=SupportContext,
    middleware=[personalized_prompt],
)

### Premium customer
Same question, but the prompt is built for a premium user.

In [4]:
resp = agent.invoke(
    {"messages": [HumanMessage(content="My order is late. What can you do?")]},
    context=SupportContext(user_name="Ada", tier="premium"),
)
resp["messages"][-1].pretty_print()

================================== Ai Message ==================================

Hello Ada! 👋 I'm so sorry to hear that your order hasn't arrived yet. I know how frustrating it is when you're eagerly waiting for something, especially as one of our valued **PREMIUM members**. Your time is precious, and we want to make this right for you immediately.

Since you're a PREMIUM customer, I have a few special options to help you right away:

1.  **Immediate Investigation**: I can personally track your package with our logistics partners to see exactly where it is and why it's delayed.
2.  **Expedited Replacement**: If the package is lost or significantly delayed, I can ship a brand-new replacement to you **overnight at no extra cost**, so you don't have to wait any longer.
3.  **Full Refund or Store Credit**: If you'd prefer, I can process a full refund instantly, or issue a **store credit with a 15% bonus** as a gesture of our apology.
4.  **Premium Compensation**: As a thank you for your p

### Free-tier customer
The exact same code path produces a different, more concise persona.

In [5]:
resp = agent.invoke(
    {"messages": [HumanMessage(content="My order is late. What can you do?")]},
    context=SupportContext(user_name="Sam", tier="free"),
)
resp["messages"][-1].pretty_print()

================================== Ai Message ==================================

Hello Sam, I'm sorry to hear your order is running late.

Since you are on our free tier, I can immediately check the current status of your shipment and provide you with the latest tracking details. If the delay is due to a carrier issue, I can also help you file a claim or contact the shipping provider directly.

Could you please share your **order number** so I can look into this for you right away?


## Summary

- `@dynamic_prompt` returns a **string** (or `SystemMessage`) used as the system prompt for that model call.
- It receives a `ModelRequest`, so it can read `request.runtime.context` and `request.state`.
- Great for personalisation, localisation, persona switching, and injecting live data such as the current date.